#Check_Availability

*Propósito:* Verificar disponibilidad de tablas fuente necesarias para construir las variables del modelo

In [0]:
%run ../config/config

In [0]:
import sys
import time
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger
from utils.control_available_info import control_available_info

In [0]:
logger = MLOpsLogger(
    name='02_check_availability',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '02_check_availability'
    }
)

In [0]:
try:
    logger.log_stage_start('check_availability', mes_vta=MES_VTA, environment=ENV)
    start_time = time.time()
    spark = SparkSession.builder.getOrCreate()

    num_errors = 0

    # ==========================================================
    # 1. ARCHIVO DE CONTROL (available_info.txt)
    # ==========================================================
    logger.info("=" * 60)
    logger.info("VERIFICANDO DISPONIBILIDAD DE TABLAS FUENTE")
    logger.info("=" * 60)

    mapping_file = os.path.join(MAPPING_PATH, "available_info.txt")
    if os.path.exists(mapping_file):
        logger.info(f"Usando archivo de control: {mapping_file}")
        reporte_disp, num_errors_ctrl = control_available_info(
            in_file_path=mapping_file,
            in_month_ref=str(MES_VTA)
        )
        logger.info(f"Revisión de archivo de control: {num_errors_ctrl} errores")
        num_errors += num_errors_ctrl
    else:
        logger.info("Archivo available_info.txt no encontrado. Se omitirá validación por archivo.")

    # ==========================================================
    # 2. VERIFICAR TABLA FUENTE PRINCIPAL
    # ==========================================================
    logger.info("\n" + "=" * 60)
    logger.info("VERIFICANDO TABLA FUENTE PRINCIPAL")
    logger.info("=" * 60)
    logger.info(f"Tabla: {SOURCE_TABLE}")
    logger.info(f"Mes a verificar: {MES_VTA}")

    try:
        count_fuente = spark.sql(f"""
            SELECT COUNT(*) as cnt
            FROM {SOURCE_TABLE}
            WHERE codmes = {MES_VTA}
            AND TRIM(codproductonivel0rbm) IN ('PYME REVOLVENTE', 'PYME NO REVOLVENTE')
            AND codclaveunicocli IS NOT NULL
        """).collect()[0]['cnt']

        if count_fuente == 0:
            logger.warning(f"⚠️ Tabla fuente no tiene datos para codmes={MES_VTA} (con filtro PYME)")
            num_errors += 1
        else:
            logger.info(f"✓ Tabla fuente disponible: {count_fuente:,} registros para mes {MES_VTA}")

    except AnalysisException as e:
        logger.error(f"Tabla fuente no existe o no accesible: {str(e)}")
        num_errors += 1
    except Exception as e:
        logger.error(f"Error inesperado al consultar tabla fuente: {str(e)}")
        num_errors += 1

    # ==========================================================
    # 3. VERIFICAR MODELO EN MLFLOW
    # ==========================================================
    logger.info("\n" + "=" * 60)
    logger.info("VERIFICANDO MODELO REGISTRADO EN MLFLOW")
    logger.info("=" * 60)

    if REGISTERED_MODEL_NAME and REGISTERED_MODEL_NAME.strip():
        try:
            import mlflow
            from mlflow.tracking import MlflowClient
            import logging
            # Silenciar temporalmente los logs de grpc para evitar el traceback largo
            grpc_logger = logging.getLogger('grpc')
            original_level = grpc_logger.level
            grpc_logger.setLevel(logging.ERROR)

            try:
                client = MlflowClient()
                latest = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "latest-databricks")
                logger.info(f"✓ Modelo disponible: {REGISTERED_MODEL_NAME} v{latest.version}")
            except Exception as e_inner:
                if "CONFIG_NOT_AVAILABLE" in str(e_inner) or "modelRegistryUri" in str(e_inner):
                    logger.warning("⚠️ MLflow Model Registry no configurado en este clúster. La verificación de MLflow no está disponible.")
                    logger.warning("  Para habilitarla, agrega 'spark.mlflow.modelRegistryUri = databricks-uc' en la configuración Spark del clúster.")
                elif "RESOURCE_DOES_NOT_EXIST" in str(e_inner):
                    logger.warning(f"Modelo '{REGISTERED_MODEL_NAME}' no encontrado en MLflow (aún no registrado).")
                else:
                    logger.warning(f"No se pudo verificar modelo en MLflow: {e_inner}")
            finally:
                grpc_logger.setLevel(original_level)

        except ImportError:
            logger.info("MLflow no instalado.")
        except Exception as e:
            logger.warning(f"Error general al verificar MLflow: {e}")
    else:
        logger.info("REGISTERED_MODEL_NAME no definido. No se verificará MLflow.")

    # ==========================================================
    # 4. RESUMEN Y TASK VALUES
    # ==========================================================
    duration = time.time() - start_time

    dbutils.jobs.taskValues.set(key="num_errors", value=num_errors)
    dbutils.jobs.taskValues.set(key="count_fuente", value=int(count_fuente) if 'count_fuente' in locals() else 0)

    logger.log_data_quality(
        dataset_name='availability_check',
        total_records=int(count_fuente) if 'count_fuente' in locals() else 0,
        valid_records=int(count_fuente) if 'count_fuente' in locals() and num_errors == 0 else 0,
        invalid_records=num_errors,
        errors_found=num_errors
    )

    logger.log_stage_end('check_availability', duration=duration, num_errors=num_errors,
                        count_fuente=int(count_fuente) if 'count_fuente' in locals() else 0)

    if num_errors > 0:
        raise ValueError(f"Verificación falló con {num_errors} error(es). Revisar logs.")

    logger.info("=" * 60)
    logger.info(f"✓ DISPONIBILIDAD VERIFICADA EXITOSAMENTE en {duration:.2f}s")
    logger.info("=" * 60)

except Exception as e:
    logger.log_exception(operation='check_availability', exception=e, should_raise=True,
                            mes_vta=MES_VTA, environment=ENV)

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()